# Métricas de Avaliação para Clustering

Avaliar um agrupamento é mais difícil do que avaliar um classificador. Existem dois cenários. Sem rótulos, resta inferir a qualidade a partir da própria geometria dos dados: são as **métricas internas**, que medem coesão e separação dos grupos. Com rótulos de referência (em datasets sintéticos ou benchmarks), dá para comparar diretamente a partição encontrada com a real: são as **métricas externas**.

Neste notebook não implementamos nenhuma métrica do zero. O foco é saber usar as versões das bibliotecas e, principalmente, entender o que cada uma assume.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.datasets import make_blobs, make_circles, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (silhouette_score, silhouette_samples, davies_bouldin_score,
                             rand_score, adjusted_rand_score)
from sklearn.metrics.cluster import contingency_matrix
from hdbscan.validity import validity_index

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## Datasets

Usaremos três conjuntos com topologias deliberadamente diferentes, todos padronizados. O **blobs** traz clusters esféricos e bem separados, o cenário ideal para métodos baseados em centróides. O **moons** traz duas luas entrelaçadas, alongadas e não convexas. E o **circles** traz dois anéis concêntricos, clusters não convexos que compartilham o mesmo centro.

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, noise=0.08, factor=0.4, random_state=42)

datasets = {
    'Blobs':   (StandardScaler().fit_transform(X_blobs), y_blobs),
    'Moons':   (StandardScaler().fit_transform(X_moons), y_moons),
    'Circles': (StandardScaler().fit_transform(X_circles), y_circles),
}

In [ ]:
def plot_clusters(X, labels, ax=None, title=None):
    """Desenha os clusters, com o ruído do DBSCAN (-1) em preto."""
    if ax is None:
        ax = plt.gca()

    unique_labels = np.unique(labels)
    colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))

    for color, label in zip(colors, unique_labels):
        mask = labels == label
        ax.scatter(X[mask, 0], X[mask, 1],
                   color='black' if label == -1 else color,
                   s=40, alpha=0.8, edgecolor='k', linewidth=0.3)

    if title:
        ax.set_title(title)

    return ax


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, (X, y)) in zip(axes, datasets.items()):
    plot_clusters(X, y, ax=ax, title=name)

plt.suptitle('Estrutura Real dos Datasets', fontsize=16)
plt.tight_layout()
plt.show()

## Aplicação dos Algoritmos

Rodamos o K-Means e o DBSCAN em cada dataset. O K-Means recebe o número real de clusters; o DBSCAN recebe `eps` e `min_samples` e descobre o número sozinho.

In [ ]:
clustering_results = {}

for name, (X, y) in datasets.items():
    n_clusters = len(np.unique(y))

    clustering_results[name] = {
        'K-Means': KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit_predict(X),
        'DBSCAN': DBSCAN(eps=0.4, min_samples=5).fit_predict(X),
    }

In [ ]:
fig, axes = plt.subplots(len(datasets), 3, figsize=(18, 15))

for i, (dataset_name, results) in enumerate(clustering_results.items()):
    X, y = datasets[dataset_name]

    plot_clusters(X, y, ax=axes[i, 0], title='Real' if i == 0 else None)
    axes[i, 0].set_ylabel(dataset_name, fontsize=14, fontweight='bold')

    for j, (algo_name, labels) in enumerate(results.items(), start=1):
        plot_clusters(X, labels, ax=axes[i, j], title=algo_name if i == 0 else None)

plt.suptitle('Resultados dos Algoritmos', fontsize=16)
plt.tight_layout()
plt.show()

Visualmente o diagnóstico é imediato: nos blobs os dois acertam; nas luas e nos círculos, apenas o DBSCAN recupera a estrutura real, enquanto o K-Means corta os grupos com fronteiras retas que ignoram a forma dos dados. Guarde essa impressão, porque algumas métricas vão discordar dela.

## Métricas Internas

As métricas internas usam apenas `X` e os rótulos encontrados, sem nenhuma referência externa. Todas seguem a mesma assinatura:

```python
metrica(X, labels)
```

### Silhouette

O Silhouette compara, para cada ponto, o quanto ele está próximo dos colegas de cluster com o quanto está próximo do cluster vizinho mais próximo. Varia de -1 a +1, e **quanto maior, melhor**.

$$ s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}} $$

- $a(i)$ é a distância média de $i$ aos demais pontos do seu cluster (coesão).
- $b(i)$ é a menor distância média de $i$ a um cluster do qual não faz parte (separação).

O `silhouette_score` devolve a média de $s(i)$ sobre todos os pontos; o `silhouette_samples` devolve o valor de cada ponto.

In [ ]:
for dataset_name, results in clustering_results.items():
    X, _ = datasets[dataset_name]

    for algo_name, labels in results.items():
        score = silhouette_score(X, labels)
        print(f"{dataset_name:8} {algo_name:8} {score:7.3f}")

Nas luas e nos círculos o Silhouette **prefere o K-Means**, que erra a estrutura. O motivo fica claro olhando os valores por ponto nos círculos:

In [ ]:
X, _ = datasets['Circles']

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, (algo_name, labels) in zip(axes, clustering_results['Circles'].items()):
    s = silhouette_samples(X, labels)
    sc = ax.scatter(X[:, 0], X[:, 1], c=s, cmap='RdYlGn', vmin=-1, vmax=1,
                    s=40, edgecolor='k', linewidth=0.3)
    ax.set_title(f'{algo_name}: silhouette médio {s.mean():.3f}')

fig.colorbar(sc, ax=axes, label='s(i)')
plt.show()

O Silhouette usa distâncias médias. Num anel, o ponto diametralmente oposto é colega de cluster e está longe, inflando o $a(i)$; o anel vizinho logo ao lado está perto e deprime o $b(i)$. No DBSCAN, os pontos do anel externo ficam perto de zero ou negativos, e a silhueta desaba mesmo com a partição perfeita.

Um detalhe prático: o Scikit-Learn trata o rótulo -1 do DBSCAN como mais um cluster. Se quiser ignorar o ruído, filtre antes: `silhouette_score(X[labels != -1], labels[labels != -1])`.

### Davies-Bouldin

O Davies-Bouldin resume cada cluster a duas quantidades: a dispersão em torno do seu centróide e a distância até os outros centróides. Para cada cluster, toma o pior par possível; o índice é a média desses piores casos. Começa em 0 e **quanto menor, melhor**.

$$ DBI = \frac{1}{K} \sum_{k=1}^{K} \max_{m \neq k} \frac{S_k + S_m}{\|\mu_k - \mu_m\|} $$

onde $S_k$ é a distância média dos pontos do cluster $k$ ao seu centróide $\mu_k$.

In [ ]:
for dataset_name, results in clustering_results.items():
    X, _ = datasets[dataset_name]

    for algo_name, labels in results.items():
        score = davies_bouldin_score(X, labels)
        print(f"{dataset_name:8} {algo_name:8} {score:7.3f}")

Nos círculos o DBSCAN, que acerta, recebe um índice absurdo. Os dois anéis são concêntricos, então seus centróides praticamente coincidem, e a distância entre eles está no denominador. O valor não diz que o agrupamento é ruim, diz que a suposição do índice (clusters representáveis por um centróide) não vale para esses dados.

### DBCV

O **Density-Based Clustering Validation** (Moulavi et al., 2014) foi criado justamente para avaliar clusters de forma arbitrária. Em vez de centróides ou distâncias médias, ele olha para a densidade:

- a **esparsidade** de um cluster é a maior aresta da árvore geradora mínima construída dentro dele, isto é, o maior "vão" que é preciso atravessar para percorrê-lo;
- a **separação** entre dois clusters é a menor distância de densidade entre eles.

Cada cluster recebe um valor entre -1 e +1 comparando separação e esparsidade, e o índice é a média ponderada pelo tamanho. **Quanto maior, melhor.** Os pontos de ruído não entram em nenhum cluster, mas penalizam o score, já que os pesos são tamanho do cluster sobre o total de pontos.

O Scikit-Learn não tem o DBCV; a implementação mais usada está no pacote `hdbscan` (`pip install hdbscan`), que exige `X` em `float64`.

In [ ]:
for dataset_name, results in clustering_results.items():
    X, _ = datasets[dataset_name]

    for algo_name, labels in results.items():
        score = validity_index(X.astype(np.float64), labels)
        print(f"{dataset_name:8} {algo_name:8} {score:7.3f}")

Agora a ordem se inverte: nas luas e nos círculos o DBCV prefere o DBSCAN, e o K-Means recebe valores bem negativos, porque seus clusters atravessam regiões vazias. Os valores absolutos do DBSCAN não são altos (nas luas chega a ser levemente negativo, já que as pontas de uma lua ficam próximas da outra), mas a ordem entre partições do mesmo dataset está correta. Assim como as demais métricas internas, o DBCV serve para comparar agrupamentos de um mesmo conjunto, não para ser lido isoladamente.

O custo é maior: o DBCV precisa das distâncias entre todos os pares de pontos de cada cluster, $O(N^2)$.

In [ ]:
internal_results = []

for dataset_name, results in clustering_results.items():
    X, _ = datasets[dataset_name]

    for algo_name, labels in results.items():
        silhouette = silhouette_score(X, labels)
        davies_bouldin = davies_bouldin_score(X, labels)
        dbcv = validity_index(X.astype(np.float64), labels)

        internal_results.append({
            'Dataset': dataset_name,
            'Algoritmo': algo_name,
            'Silhouette ↑': silhouette,
            'Davies-Bouldin ↓': davies_bouldin,
            'DBCV ↑': dbcv,
        })

pd.DataFrame(internal_results).round(3)

A lição: **Silhouette e Davies-Bouldin embutem a hipótese de clusters convexos e isotrópicos**, que é exatamente a hipótese do K-Means. Usá-los para escolher entre algoritmos tende a premiar quem faz essa suposição, valha ela ou não nos dados. Escolher a métrica interna é escolher a hipótese.

## Métricas Externas

Como os datasets são sintéticos, temos os rótulos verdadeiros e podemos medir diretamente a concordância entre a partição encontrada e a real. As métricas externas não usam `X`, apenas os dois vetores de rótulos:

```python
metrica(y_true, labels)
```

Os números dos clusters são arbitrários: o cluster 0 do algoritmo não precisa corresponder à classe 0. Por isso nenhuma dessas métricas compara rótulo com rótulo, como faria a acurácia.

### Matriz de Contingência

É a base de todas as métricas externas. A célula $(i, j)$ conta quantos pontos da classe real $i$ caíram no cluster $j$. Um agrupamento perfeito tem exatamente um valor não nulo em cada linha e em cada coluna (a ordem das colunas não importa).

In [ ]:
fig, axes = plt.subplots(len(datasets), 2, figsize=(9, 12))

for i, (dataset_name, results) in enumerate(clustering_results.items()):
    _, y = datasets[dataset_name]

    for j, (algo_name, labels) in enumerate(results.items()):
        cm = contingency_matrix(y, labels)
        ax = axes[i, j]
        ax.imshow(cm, cmap='Blues', vmin=0)

        for (r, c), value in np.ndenumerate(cm):
            ax.text(c, r, value, ha='center', va='center',
                    color='white' if value > cm.max() / 2 else 'black')

        ax.set_xticks(range(cm.shape[1]), np.unique(labels))
        ax.set_yticks(range(cm.shape[0]), np.unique(y))
        ax.set_xlabel('Cluster')
        ax.set_ylabel(f'{dataset_name}\nClasse real' if j == 0 else 'Classe real')
        ax.grid(False)
        if i == 0:
            ax.set_title(algo_name)

plt.tight_layout()
plt.show()

Nos blobs as duas matrizes são permutações da diagonal. Nos círculos, o K-Means espalha cada anel quase igualmente pelos dois clusters, enquanto o DBSCAN coloca cada anel inteiro num cluster só. Se houvesse ruído, o rótulo -1 apareceria como uma coluna a mais.

### Pureza

A pureza atribui a cada cluster a classe majoritária dentro dele e mede a fração de pontos que pertencem a essa classe. Vai de 0 a 1, e **quanto maior, melhor**.

$$ \text{Pureza} = \frac{1}{N} \sum_{k} \max_{j} |C_k \cap T_j| $$

Não há função pronta no Scikit-Learn, mas ela sai direto da matriz de contingência: o máximo de cada coluna (cluster), somado e dividido pelo total.

In [ ]:
def purity_score(y_true, labels):
    cm = contingency_matrix(y_true, labels)
    majority = cm.max(axis=0)
    return majority.sum() / cm.sum()


for dataset_name, results in clustering_results.items():
    _, y = datasets[dataset_name]

    for algo_name, labels in results.items():
        score = purity_score(y, labels)
        print(f"{dataset_name:8} {algo_name:8} {score:7.3f}")

A pureza tem um defeito grave: ela só olha para dentro de cada cluster, então fragmentar sempre ajuda. No extremo, um cluster por ponto dá pureza 1. Basta diminuir o `eps` do DBSCAN nos círculos para ver o efeito:

In [ ]:
X, y = datasets['Circles']
fragmented = DBSCAN(eps=0.3, min_samples=5).fit_predict(X)

n_clusters = len(np.unique(fragmented[fragmented != -1]))
purity = purity_score(y, fragmented)

print(f"clusters: {n_clusters}")
print(f"pureza:   {purity:.3f}")

Com dez clusters a pureza continua praticamente perfeita, embora os anéis tenham sido picotados. Ela precisa ser lida junto do número de clusters, ou substituída por uma métrica que penalize a fragmentação.

### Rand Index

O Rand Index olha para **pares de pontos**. Para cada par, pergunta se as duas partições concordam: ou ambas colocam os dois pontos juntos, ou ambas os separam. O índice é a fração de pares em que há concordância. Vai de 0 a 1, e **quanto maior, melhor**.

$$ RI = \frac{a + b}{\binom{N}{2}} $$

- $a$: pares que estão juntos na partição real e no agrupamento;
- $b$: pares que estão separados na partição real e no agrupamento.

Como conta tanto os pares juntos quanto os separados, penaliza a fragmentação e também a fusão.

In [ ]:
for dataset_name, results in clustering_results.items():
    _, y = datasets[dataset_name]

    for algo_name, labels in results.items():
        score = rand_score(y, labels)
        print(f"{dataset_name:8} {algo_name:8} {score:7.3f}")

_, y = datasets['Circles']
score = rand_score(y, fragmented)
print(f"\nCircles fragmentado: {score:.3f}")

O problema do Rand Index é o piso. Nos círculos, o K-Means é tão bom quanto um sorteio e mesmo assim tira 0,498. Com duas classes equilibradas, metade dos pares concorda por acaso; com muitos clusters, a maioria dos pares está separada em ambas as partições e o índice fica próximo de 1 para quase qualquer agrupamento. O valor bruto é difícil de interpretar.

### Adjusted Rand Index (ARI)

O ARI corrige o Rand Index pelo valor que se esperaria de um agrupamento aleatório com os mesmos tamanhos de cluster:

$$ ARI = \frac{RI - \mathbb{E}[RI]}{\max(RI) - \mathbb{E}[RI]} $$

Vale 1 para partições idênticas, fica em torno de 0 para um agrupamento aleatório e pode ser negativo para um pior que o acaso. É a métrica externa mais usada.

In [ ]:
for dataset_name, results in clustering_results.items():
    _, y = datasets[dataset_name]

    for algo_name, labels in results.items():
        score = adjusted_rand_score(y, labels)
        print(f"{dataset_name:8} {algo_name:8} {score:7.3f}")

_, y = datasets['Circles']
score = adjusted_rand_score(y, fragmented)
print(f"\nCircles fragmentado: {score:.3f}")

random_labels = np.random.default_rng(42).integers(0, 2, len(y))
score = adjusted_rand_score(y, random_labels)
print(f"Circles aleatório:   {score:.3f}")

Agora a leitura é direta: o K-Means nos círculos é indistinguível de um sorteio, o DBSCAN acerta tudo e o DBSCAN fragmentado fica no meio do caminho, bem longe da pureza quase perfeita que recebia.

In [ ]:
external_results = []

for dataset_name, results in clustering_results.items():
    _, y = datasets[dataset_name]

    for algo_name, labels in results.items():
        purity = purity_score(y, labels)
        rand = rand_score(y, labels)
        ari = adjusted_rand_score(y, labels)

        external_results.append({
            'Dataset': dataset_name,
            'Algoritmo': algo_name,
            'Pureza ↑': purity,
            'Rand ↑': rand,
            'ARI ↑': ari,
        })

pd.DataFrame(external_results).round(3)

As métricas externas confirmam a leitura visual e o DBCV, e contradizem Silhouette e Davies-Bouldin nas luas e nos círculos. Só que esse árbitro não existe num problema real: se tivéssemos os rótulos, não estaríamos clusterizando. É por isso que as métricas internas são necessárias, e é por isso que é preciso saber o que elas assumem antes de confiar nelas.

## Exercícios

### Exercício 1: Hierárquico e GMM

Adicione ao `clustering_results` o agrupamento hierárquico (`AgglomerativeClustering`) e o GMM (`GaussianMixture`), ambos com o número real de clusters, e recalcule todas as métricas internas e externas nos três datasets.

No hierárquico, compare o `linkage='ward'` com o `linkage='single'`. Quais algoritmos as métricas internas tratam como tratam o K-Means? Algum outro algoritmo além do DBSCAN recupera as luas e os círculos, e como Silhouette, Davies-Bouldin e DBCV o avaliam?

In [ ]:
# Seu código aqui

### Exercício 2: Avaliação no Dataset Iris

Aplique o K-Means e o DBSCAN ao dataset Iris, lembrando de padronizar os dados antes, e avalie os resultados com as três métricas internas.

Em seguida, calcule pureza, Rand e ARI contra as espécies reais e compare: alguma métrica interna elege o mesmo vencedor que o ARI? Discuta o resultado à luz do que cada índice assume sobre a forma dos clusters e do fato de que duas das três espécies do Iris se sobrepõem.

In [ ]:
# Seu código aqui